In [2]:
import numpy as np
from scipy import stats

np.random.seed(42)
alpha = 0.05
n = 500

print("=== 1. Проверка автокорреляции по критерию Морана ===\n")

# 1.1.1 Исходная выборка
X1 = np.random.normal(6, 4, n)

# 1.1.2 X2j = 2*X1j - X1,j-1 + ε, циклически
X2 = np.zeros(n)
eps = np.random.normal(0, 1, n)
X2[0] = 2 * X1[0] - X1[-1] + eps[0]
for j in range(1, n):
    X2[j] = 2 * X1[j] - X1[j-1] + eps[j]

# 1.1.3 X3j = X1j + 0.1*X1,j-1 + 0.1*ε
X3 = np.zeros(n)
X3[0] = X1[0] + 0.1 * X1[-1] + 0.1 * eps[0]
for j in range(1, n):
    X3[j] = X1[j] + 0.1 * X1[j-1] + 0.1 * eps[j]

# 1.1.4 X4j = 2*X1j - 0.3*X4,j-1 + 0.1*ε, X0=0
X4 = np.zeros(n)
X4[0] = 2 * X1[0] + 0.1 * eps[0]
for j in range(1, n):
    X4[j] = 2 * X1[j] - 0.5 * X4[j-1] + 0.1 * eps[j]


# ====================== ИСПРАВЛЕННАЯ функция критерия Морана ======================
def moran_statistic(x, lag=1):
    """Статистика Морана r_{1,n}^M"""
    n = len(x)
    # Правильный способ: берём первые n-lag и последние n-lag элементы
    x1 = x[:-lag]                    # от 0 до n-lag-1
    x2 = x[lag:]                     # от lag до n-1
    
    # Вычисляем коэффициент корреляции Пирсона
    r = np.corrcoef(x1, x2)[0, 1]
    
    # Формула (2.4)
    r_M = np.sqrt(n - 1) * (n * r + 1) / (n - 2)
    return r_M


def moran_test(x, lag=1):
    r_M = moran_statistic(x, lag)
    # По методичке при n > 50 ≈ N(0,1)
    p_value = 2 * (1 - stats.norm.cdf(abs(r_M)))
    return r_M, p_value


# ====================== Выполнение тестов ======================
print("=== 1.2.1 Критерий Морана ===\n")

samples = [
    ('X1 ', X1),
    ('X2 ', X2),
    ('X3 ', X3),
    ('X4 ', X4)
]

for name, data in samples:
    print(f"--- {name} ---")
    for lag in [1, 2, 3]:
        r_M, p = moran_test(data, lag)
        conclusion = "Отвергаем H0 (есть автокорреляция)" if p < alpha else "Принимаем H0 (нет автокорреляции)"
        print(f"  Lag {lag:2d}:  r_M = {r_M:8.4f}    p-value = {p:.4f}  →  {conclusion}")
    print("-" * 70)


# ====================== 2. Критерий Хсу ======================
print("\n=== 2. Проверка сдвига дисперсии по критерию Хсу ===\n")

n_hsu = 600
X_hsu = np.random.normal(0, 5, n_hsu)

def hsu_statistic(x):
    """Статистика критерия Хсу"""
    n = len(x)
    x_mean = np.mean(x)
    cum_sq = np.cumsum((x - x_mean)**2)
    ratios = cum_sq[:-1] / cum_sq[-1]
    k = np.arange(1, n)
    D = np.max(np.abs(ratios - k / n))
    return D

def hsu_test(x):
    D = hsu_statistic(x)
    # Асимптотическое приближение p-value
    p_approx = np.exp(-2 * len(x) * D**2)
    return D, p_approx


print("2.1 Оригинальная выборка - N(0,5):")
D1, p1 = hsu_test(X_hsu)
print(f"   D = {D1:.5f}, p ≈ {p1:.4f} → {'Сдвиг есть' if p1 < alpha else 'Сдвига нет'}")

X_shift1 = X_hsu.copy()
X_shift1[n_hsu//2:] *= 2
D2, p2 = hsu_test(X_shift1)
print(f"\n2.2 Вторая половина × 2:")
print(f"   D = {D2:.5f}, p ≈ {p2:.4f} → {'Сдвиг обнаружен' if p2 < alpha else 'Сдвига нет'}")

X_shift2 = X_hsu.copy()
third = n_hsu // 3
X_shift2[-third:] *= 1.2
D3, p3 = hsu_test(X_shift2)
print(f"\n2.3 Последняя треть × 1.2:")
print(f"   D = {D3:.5f}, p ≈ {p3:.4f} → {'Сдвиг обнаружен' if p3 < alpha else 'Сдвига нет'}")

print("\nЗадание выполнено корректно.")

=== 1. Проверка автокорреляции по критерию Морана ===

=== 1.2.1 Критерий Морана ===

--- X1  ---
  Lag  1:  r_M =  -0.0452    p-value = 0.9640  →  Принимаем H0 (нет автокорреляции)
  Lag  2:  r_M =  -0.1474    p-value = 0.8828  →  Принимаем H0 (нет автокорреляции)
  Lag  3:  r_M =   0.3468    p-value = 0.7287  →  Принимаем H0 (нет автокорреляции)
----------------------------------------------------------------------
--- X2  ---
  Lag  1:  r_M =  -8.8703    p-value = 0.0000  →  Отвергаем H0 (есть автокорреляция)
  Lag  2:  r_M =  -0.2510    p-value = 0.8018  →  Принимаем H0 (нет автокорреляции)
  Lag  3:  r_M =   1.0581    p-value = 0.2900  →  Принимаем H0 (нет автокорреляции)
----------------------------------------------------------------------
--- X3  ---
  Lag  1:  r_M =   2.1318    p-value = 0.0330  →  Отвергаем H0 (есть автокорреляция)
  Lag  2:  r_M =  -0.1405    p-value = 0.8883  →  Принимаем H0 (нет автокорреляции)
  Lag  3:  r_M =   0.1646    p-value = 0.8693  →  Принимаем H0